In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)  # should print "cuda"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
pip install transformers datasets accelerate

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "EleutherAI/pythia-160m"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token
model = model.to(device)
print("Model loaded.")

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "ccdv/pubmed-summarization",
    split="train",
    streaming=True
)

texts = []
for i, sample in enumerate(dataset):

    texts.append(sample["abstract"])
    if i >= 9999:
        break

print(f"Loaded {len(texts)} abstracts.")

In [ ]:
from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=128):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt"
        )

    def __len__(self):
        return self.encodings["input_ids"].shape[0]

    def __getitem__(self, idx):
        return {`
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx]
        }

ft_dataset = TextDataset(texts, tokenizer)
ft_loader = DataLoader(
    ft_dataset,
    batch_size=16,
    shuffle=True
)
print(f"Batches: {len(ft_loader)}")

In [ ]:
from torch.optim import AdamW

ft_optimizer = AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

ft_scheduler = torch.optim.lr_scheduler.LinearLR(
    ft_optimizer,
    start_factor=0.1,
    end_factor=1.0,
    total_iters=200
)

ft_epochs = 3

model.train()

for epoch in range(ft_epochs):

    total_loss = 0

    for i, batch in enumerate(ft_loader):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=input_ids
        )

        loss = outputs.loss

        ft_optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        ft_optimizer.step()
        ft_scheduler.step()

        total_loss += loss.item()

        if i % 100 == 0:
            print(
                f"Epoch {epoch+1} | "
                f"Step {i}/{len(ft_loader)} | "
                f"Loss={loss.item():.4f}"
            )

    print(
        f"Epoch {epoch+1} DONE | "
        f"Avg Loss={total_loss/len(ft_loader):.4f}"
    )

In [ ]:
texts_clean = [t for t in texts if t and len(t.strip()) > 50]
print(f"Cleaned: {len(texts_clean)} abstracts (was {len(texts)})")

ft_dataset = TextDataset(texts_clean, tokenizer)
ft_loader = DataLoader(ft_dataset, batch_size=8, shuffle=True)
print(f"Batches: {len(ft_loader)}")

In [ ]:
sample_batch = next(iter(ft_loader))
print("input_ids NaN:", torch.isnan(sample_batch["input_ids"].float()).any())
print("Sample text:", texts_clean[0][:200])

In [ ]:
from torch.optim import AdamW

model = AutoModelForCausalLM.from_pretrained(
    "EleutherAI/pythia-160m",
    torch_dtype=torch.bfloat16
)
model = model.to(device)

ft_optimizer = AdamW(
    model.parameters(),
    lr=1e-5,
    weight_decay=0.01
)

ft_epochs = 3
model.train()

for epoch in range(ft_epochs):

    total_loss = 0
    valid_steps = 0

    for i, batch in enumerate(ft_loader):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        ft_optimizer.zero_grad()

        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=input_ids
            )
            loss = outputs.loss

        if torch.isnan(loss):
            continue

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        ft_optimizer.step()

        total_loss += loss.item()
        valid_steps += 1

        if i % 100 == 0:
            print(
                f"Epoch {epoch+1} | "
                f"Step {i}/{len(ft_loader)} | "
                f"Loss={loss.item():.4f}"
            )

    print(
        f"Epoch {epoch+1} DONE | "
        f"Avg Loss={total_loss/max(valid_steps,1):.4f}"
    )

In [ ]:
model.save_pretrained(
    "/content/drive/MyDrive/pythia160m_pubmed"
)
tokenizer.save_pretrained(
    "/content/drive/MyDrive/pythia160m_pubmed"
)
print("Fine-tuned model saved.")